# Phase 5D - Raw material analysis

Clustering and ore grade classification on the Tethys bauxite geochemistry compilation (1,093 samples). An A/S ratio above 4 is treated as acceptable for the Bayer process.

## 1. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import silhouette_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
import sklearn; print(f"Scikit-learn: {sklearn.__version__}")

SEED = 42
np.random.seed(SEED)

In [ ]:
# Upload tethys_bauxite_geochemistry.csv
from google.colab import files
print("Upload: tethys_bauxite_geochemistry.csv")
uploaded = files.upload()

## 2. Data Loading & Inspection

In [ ]:
df = pd.read_csv('tethys_bauxite_geochemistry.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSource sheets: {df['source_sheet'].value_counts().to_dict()}")
df.head()

In [ ]:
# Identify major oxide columns
oxide_cols = ['Al2O3', 'SiO2', 'Fe2O3', 'TiO2', 'CaO', 'MgO', 'K2O', 'Na2O']
available_oxides = [c for c in oxide_cols if c in df.columns]
print(f"Available oxide columns: {available_oxides}")

# Check for A/S ratio
if 'A/S' not in df.columns and 'Al2O3' in df.columns and 'SiO2' in df.columns:
    df['A/S'] = df['Al2O3'] / df['SiO2']
    print("Computed A/S ratio")

print(f"\nMissing values:")
print(df[available_oxides + ['A/S']].isnull().sum())

In [ ]:
# Summary statistics
print("Major oxide statistics:")
df[available_oxides + ['A/S']].describe()

## 3. Data Cleaning

In [ ]:
# Drop rows with missing Al2O3 or SiO2 (critical for Bayer process)
df_clean = df.dropna(subset=['Al2O3', 'SiO2']).copy()
print(f"After dropping Al2O3/SiO2 nulls: {df_clean.shape}")

# Fill remaining oxide NaNs with median
for col in available_oxides:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print(f"\nCleaned shape: {df_clean.shape}")
print(f"Remaining nulls: {df_clean[available_oxides].isnull().sum().sum()}")

## 4. Exploratory Data Analysis

In [ ]:
# A/S ratio distribution (critical threshold is ~4.0)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_clean['A/S'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(4.0, color='red', linestyle='--', linewidth=2, label='A/S = 4.0 threshold')
axes[0].set_xlabel('A/S Ratio')
axes[0].set_ylabel('Count')
axes[0].set_title('A/S Ratio Distribution')
axes[0].legend()

axes[1].scatter(df_clean['Al2O3'], df_clean['SiO2'], c=df_clean['A/S'], 
                cmap='RdYlGn', alpha=0.6, s=30)
axes[1].set_xlabel('Al2O3 (wt%)')
axes[1].set_ylabel('SiO2 (wt%)')
axes[1].set_title('Al2O3 vs SiO2 (colored by A/S)')
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('A/S Ratio')

plt.tight_layout()
plt.show()

# Ore quality breakdown
high_grade = (df_clean['A/S'] >= 6.0).sum()
medium_grade = ((df_clean['A/S'] >= 4.0) & (df_clean['A/S'] < 6.0)).sum()
low_grade = (df_clean['A/S'] < 4.0).sum()
print(f"\nOre grade distribution:")
print(f"  High grade (A/S ≥ 6.0): {high_grade} ({100*high_grade/len(df_clean):.1f}%)")
print(f"  Medium grade (4.0 ≤ A/S < 6.0): {medium_grade} ({100*medium_grade/len(df_clean):.1f}%)")
print(f"  Low grade (A/S < 4.0): {low_grade} ({100*low_grade/len(df_clean):.1f}%)")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df_clean[available_oxides + ['A/S']].corr(), 
            annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Major Oxide Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Unsupervised Clustering

In [ ]:
# Prepare data for clustering
X = df_clean[available_oxides].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA for visualization
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_}")
print(f"Cumulative: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Elbow method for optimal k
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(K_range, inertias, marker='o')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouettes, marker='o', color='tab:orange')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"\nBest k by silhouette score: {best_k}")

In [ ]:
# Final clustering with optimal k
optimal_k = 4  # adjust based on elbow/silhouette
kmeans = KMeans(n_clusters=optimal_k, random_state=SEED, n_init=20)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\nCluster sizes:")
print(df_clean['cluster'].value_counts().sort_index())

# Visualize clusters in PCA space
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df_clean['cluster'], 
                     cmap='tab10', alpha=0.6, s=40)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title(f'Bauxite Clusters (k={optimal_k})')
plt.colorbar(scatter, label='Cluster')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles
cluster_profiles = df_clean.groupby('cluster')[available_oxides + ['A/S']].mean()
print("\nCluster geochemical profiles:")
print(cluster_profiles.round(2))

## 6. Ore Grade Classification

Train a classifier to predict ore grade from major oxide composition.

In [ ]:
# Create grade labels
def classify_grade(as_ratio):
    if as_ratio >= 6.0:
        return 'high'
    elif as_ratio >= 4.0:
        return 'medium'
    else:
        return 'low'

df_clean['ore_grade'] = df_clean['A/S'].apply(classify_grade)
print(f"Ore grade distribution:")
print(df_clean['ore_grade'].value_counts())

In [ ]:
# Train classifier
X_clf = df_clean[available_oxides].values
y_clf = df_clean['ore_grade'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEED, stratify=y_clf
)

rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED)
rf_clf.fit(X_train, y_train)

# Evaluate
train_score = rf_clf.score(X_train, y_train)
test_score = rf_clf.score(X_test, y_test)
print(f"\nOre Grade Classifier Performance:")
print(f"  Train accuracy: {train_score:.3f}")
print(f"  Test accuracy: {test_score:.3f}")

y_pred = rf_clf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance
importances = rf_clf.feature_importances_
feature_imp = pd.DataFrame({
    'feature': available_oxides,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(feature_imp['feature'], feature_imp['importance'])
plt.xlabel('Feature Importance')
plt.title('Ore Grade Classification — Feature Importance')
plt.tight_layout()
plt.show()

print("\nFeature importance:")
print(feature_imp.to_string(index=False))

## 7. Export Models

In [ ]:
import pickle

raw_material_models = {
    'clustering': {
        'kmeans': kmeans,
        'scaler': scaler,
        'pca': pca,
        'n_clusters': optimal_k,
        'cluster_profiles': cluster_profiles.to_dict(),
    },
    'ore_grade_classifier': {
        'model': rf_clf,
        'feature_names': available_oxides,
        'grade_labels': ['high', 'medium', 'low'],
        'test_accuracy': test_score,
    },
    'metadata': {
        'n_samples': len(df_clean),
        'dataset': 'Tethys bauxite geochemical compilation',
        'seed': SEED,
    }
}

with open('raw_material_analysis_models.pkl', 'wb') as f:
    pickle.dump(raw_material_models, f)

print("Models saved to raw_material_analysis_models.pkl")

# Download
from google.colab import files
files.download('raw_material_analysis_models.pkl')

In [ ]:
# Save enriched dataset with cluster assignments
df_clean[['Al2O3', 'SiO2', 'Fe2O3', 'TiO2', 'CaO', 'MgO', 'A/S', 
          'cluster', 'ore_grade']].to_csv('tethys_enriched.csv', index=False)
print("Enriched dataset saved to tethys_enriched.csv")
files.download('tethys_enriched.csv')

## 8. Summary

| Component | Output |
|---|---|
| Dataset | 1,093 bauxite samples (Tethys compilation) |
| Clustering | K-means, k=4 |
| Ore grading | Random forest classifier (high/medium/low) |
| Test accuracy | see the run output above |